학습 목표
- UserChangeForm을 상속받아 회원정보 수정 폼을 만든다.
- PasswordChangeForm을 사용하여 비밀번호 변경 기능을 구현한다.
- update_session_auth_hash 함수로 비밀번호 변경 후 세션을 유지한다.
- 단방향 해시를 통한 비밀번호 암호화의 필요성의 필요성을 안다.
- 솔트와 키 스트레칭을 통한 암호화 강화 원리를 안다.
- Django의 비밀번호 저장 방식(알고리즘, 솔트 등)을 안다.
- Django 내장 auth URL을 활용해 비밀번호 초기화를 구현한다.

회원정보 수정

회원정보 수정
- User 객체를 Update 하는 과정
- 수정할 대상 User객체를 가져오고, 입력받은 새로운 정보로 기존 내용을 갱신

회원정보 수정 기능 구현
- 회원정보 수정 경로 url 생성

- [복습] 커스텀 유저 모델을 사용을 하기 위해서 Form을 다시 작성
- Custom User model을 사용할 수 있도록 상속 후 일부분만 재작성

- .../accounts/update/ url로 요청이 들어왔을 때 실행할 update 함수 작성
- 회원종보 수정에 사용할 데이터를 입력 받는 CustomUserChangeForm built-in form 사용

- 회원가입을 위해 작성한 정보를 서버에 안전하게 전송하기 위해 'POST 방식'을 사용
- 서버로부터 전달받은 CustomUserForm을 화면에 출력

- 메인 페이지에서 회원정보 수정 페이지에 접근할 수 있는 테그 생성

UserChangeForm()
- 회원정보 수정 시 사용자 입력 데이터를 받는 built-in ModelForm
- ModelForm이기 때문에, 유효성 검사를 통과한 데이터로 기존 User 객체의 내용을 갱신하고 저장(주로 관리자 페이지에서 사용)

UserChangeForm 사용 시 문제점
- User 모델의 모든 정보들(fields)까지 모두 출력됨
- 일반 사용자들이 접근하면 안되는 정보는 출력하지 않아야 함

> CustomUserChangeForm에서 출력 필드를 다시 조정하기

CustomUserChangeForm 출력 필드 재정의
- User Model의 필드 목록 확인

회원정보 수정 로직 완성

비밀번호 변경
- 인증된 사용자의 Session 데이터를 Update 하는 과정
- 기존 비밀번호를 통해 사용자를 인증하고, 새로운 비밀번호를 암호화하여 갱신

비밀번호 변경 기능 구현 작성
- django는 비밀번호 변경 페이지를 회원정보 수정 form 하단에서 별도 주소로 안내

- django에서 안내하는 비밀번호 변경 URL에 맞춰서 작성

- .../accounts/password/ url로 요청이 들어올 때 실행할 password함수 작성
- 비밀번호 변경에 사용할 데이터를 입력 받는 PasswordChangeForm built-in form 사용



PasswordChangeForm()
- 비밀번호 변경 시 사용자 입력 데이터를 받는 built-in Form
- 일반 'Form'이며, 유효성 검사(기존 비밀번호 확인, 새 비밀번호 일치 여부)를 통과한 데이터로 사용자의 비밀번호를 안전하게 암호화하여 갱신하는 역할을 수행

세션 무효화 방지

암호 변경 시 세션 무효화
- 비밀번호가 변경되면 기존 세션과의 회원 인증 정보가 일치하지 않게 되어 버려 로그인 상태가 유지되지 못하고 로그아웃 처리됨
- 비밀번호가 변경되면서 기존 세션과의 회원 인증 정보가 일치하지 않기 때문

> 암호 변경 시 세션 무효화를 막는 방법은?

암호 변경 시 세션 무효화를 막아주는 함수  
update_session_auth_hash(request, user)  

- 암호가 병경되면 새로운 password의 Session Data로 기존 session을 자동으로 갱신
- update_session_auth_hash를 password 함수에 적용

비밀번호 암호화

암호화의 중요성
- 많은 해킹사태가 발생하고 있고, 데이터가 유출되더라도 그 내용을 알 수 없도록 하는 암호화는 특히 중요

우리가 사용하는 비밀번호는 어떻게 저장되고 있을까?

1. 사용자가 입력한 비밀번호 그대로 저장하는 방식 => 보안에 매우 취약하므로 X
   - 데이터베이스가 해킹 당하면, 공격자는 아이디와 비밀번호 목록을 그대로 손에 넣게 되고, 이를 통해 직접 로그인하여 개인정보, 금융 정보, 주소록 등 모든 데이터를 유출하거나 서비스를 악용할 수 있음
   - 악의적인 내부 직원이 데이터베이스에 접근하여 모든 사용자으 비밀번호를 볼 수 있음
   - 대부분의 사람들은 여러 서비스에서 동일한 아이디와 비밀번호 사용하기에 탈취한 정보를 이용해 다른 사이트에 그대로 대입하여 2차 피해를 발생시킴(Credental Stuffing 공격)

2. 일정한 규칙에 따라 비밀번호를 알아볼 수 없는 문자로 '인코딩'한 후 저장 => 보안에 매우 취약하므로 X
   - 데이터베이스에 알아볼 수 없는 문자로 저장되어 있다고 하더라도, 인코딩은 비밀키 없이도 정해진 규칙에 따라 누구나 원래의 값으로 되돌릴 수 있음
   - 공격자는 아주 간단한 디코딩 작업만으로 모든 사용자의 실제 비밀번호를 즉시 알아낼 수 있음
   - 사실상 비밀번호를 평문으로 저장하는 것과 동일한 수준의 위험을 초래

```
- 인코딩 : 정보를 표현하는 형식을 다른 형식으로 변환하는 과정
- 비밀키 : 암호화된 데이터를 풀거나, 디지털 서명을 만들 때 쓰는 열쇠
```

3. 비밀번호를 복원이 불가능한 고정된 길이의 문자열로 변환 후 저장 => 보안에 필수
   - 데이터베이스가 유출되어도 공격자는 복잡하게 얽힌 문자열을 보게 되고, 복원이 불가능하기 때문에 실제 비밀번호를 알 수 없음
   - 악의적인 내부 직원이 비밀번호를 보더라도 암호화된 비밀번호를 보게 되므로, 실제 비밀번호를 유추할 수 없음

> 비밀번호를 복원이 불가능한 고정된 길이로 바꾸는 과정을 "해쉬(hash)"라고 함

```
왜 고정된 길이의 문자열로 변환하나요?
1. 보안성 : 변환된 문자열의 길이가 다르다면, 길이만 보고도 원래 비밀번호의 길이를 유추할 수 있습니다.
3. 일관성 : 길이가 동일하기 때문에 저장 공간을 예측하고 설계하기 쉬우며, 검색/비교하는 처리 속도도 일정하게 유지할 수 있습니다.
```


해쉬 (Hash)
- 임의의 크기를 가진 데이터를 고정된 크기의 고유한 값으로 변환하는 것

```
해시는 데이터를 믹서기에 넣어 만든 주스와 같습니다.
- 아무리 복잡한 과일(데이터)이라도 한 번 갈아버리면, 절대 다시 원래의 과일로 되돌릴 수 없습니다.
- 하지만 같은 재료를 넣으면 언제나 똑같은 맛의 주스가 나옵니다.
- 이처럼 원본을 알 수 없게 변환하여 보안을 지키거나, 데이터가 변경되지 않았는지 확인하는 기술입니다.
```

해시함수(Hash function)
- 임의 길이 데이터를 입력 받아 고정 길이(정수)로 변환해 주는 함수

```
데이터를 넣으면 암호화된 문자열(해시값)을 뱉어내는 '마법의 상자'  
1. 고정된 길이: 아무리 긴 소설책을 넣어도, 짧은 단어를 넣어도 결과물의 길이는 똑같습니다.
2. 단방향성 (One-way): 결과물만 보고 원본을 추측하는 것은 불가능합니다.
3. 눈사태 효과 (Avalanche Effect): 입력 값이 점 하나만 달라져도, 결과물은 완전히 다른 값이 됩니다.
```



해쉬(hash)
- 데이터를 고정도니 크기의 값으로 변환하는 과정
- 작은 변화에도 해시값이 크게 달라지는 특성으로 인해 변조 여부를 쉽게 확인할 수 있음
- 입력값이 들어오더라도 해시 함수에 의해서 다른 값으로 바뀌며, 동일한 값은 항상 동일한 해시값을 생성

Django는 기본적으로 SHA-256 해시 함수를 사용해서 암호화
- 입력한 비밀번호의 길이와는 상관없이 동일한 길이의 해시 값을 생성
- 1글자만 다르더라도 전혀 다른 해시 값을 생성

SHA-256 (Secure Hash Algorithm - 256)
- 현재 가장 널리 사용되는 표준 암호화 해시 알고리즘

```
어떤 데이터를 넣더라도 항상 256비트(64글자)의 결과물을 만들어냅니다.
현재 가장 신뢰받는 알고리즘으로, 비트코인의 핵심 기술이자 전 세계 보안 표준을 사용됩니다.
```

해시 함수를 활용해 단방향 암호화를 하면 더 이상 문제가 없을까?
- 비밀번호를 해시 값으로 저장하면 공격자가 유출된 데이터베이스를 봐도 원래 비밀번호를 알 수 없으니 안전해 보일 수 있음
- 하지만 공격자들은 해시 값ㅇ르 미리 계산해두는 방식으로 공격을 시도

> 이 방식이 바로 '레인보우 테이블'공격

레인보우 테이블(Rainbow Table)
- 공격자가 자주 사용되는 비밀번호들을 미리 수백만, 수십억 개를 해시로 변환 해 저장대 둔 거대한 정답지
- 공격 방식
    1. 공격자가 DB를 탈취해 사용자의 비밀번호 해시 값을 얻음
    2. 해시 값을 공격자 자신의 레인보우 테이블에서 검색
    3. 테이블에서 일치되는 값을 찾아내면서, 역방향으로 되돌리지 않고 비밀번호를 알아내는 데 성공

그러면 레인보우 테이블 공격은 어떻게 방어할까?
- 공격자가 아무리 거대한 레인보우 테이블ㅇ르 가지고 있더라도, 사용자의 해시 값이 레인보우 테이블에 없도록 해시 값을 만들면 됨
- 해시 값은 입력값이 단 한 글자만 달라져도 해시값은 완전히 달라지는 눈사태 효과가 있음
- 결국 값은 비밀번호라도 사용자마다 '임의의 문자열'을 비밀번호에 붙여서 해시 암호화를 진행
  - 임의의 문자열이 추가된 상태로 해시 값을 마들기 때문에 눈사태 효과에 의해 같은 비밀번호도 다른 해시 값이 나옴

> 여기서 '임의의 문자열'역할을 하는 것이 바로 '솔트(Salt)'

솔트(salt)
- 각 사용자마다 고유하게 생성도니 임의의 문자열(솔트)을 비밀번호에 덧붙여서 해시 값을 생성
- 이 솔트(salt)는 해시 값과 함께 데이터베이스에 저장

```
공격자가 솔트(salt)값을 알아도 되나요?
- 공격자가 솔트값을 알아도 상관없음
- userA의 솔트와 해시값을 훔쳤다면, 공격자는 userA만을 위한 레인보우 테이블을 만들어야 합니다.
- 이는 '하나의 답안지로 수백만 명을 공격하는 방식'에서 '한 명을 공격하기 위해 매번 새로운 답안지를 만드는 방식'으로 바꾸면서, 공격의 효율성을 극도로 떨어뜨립니다.
```

솔트로 레인보우 테이블 공격을 막았으니, 안전할까?
- 공격자는 이제 미리 만들어 둔 답안지를 쓸 수 없어짐
- 그래서 단순하지만 확실할 방법으로써 가능한 모든 비밀번호를 하나씩 직접 대입 해보는 방식으로 공격
- 이 방식은 현대 컴퓨터의 엄청난 속도 때문에 생각보다 훨씬 위협적
  - 최신 GPU는 초당 약, 1,500억 번이상 추측할 수 있음

> 이 방식이 바루 '무차별 대입 공격(Brute-force Attack)'

무차별 대입 공격(Brute-force Attack)
- 가장 원시적이지만 강력한 방법으로써 가능한 모든 비밀번호를 하나씩 대입하는 방식
- 이 공격은 시간과의 싸움이며, 현대 컴퓨터의 빠른 연산 속도가 공격자의 무기가 됨

그러면 무차별 대입 공격은 어떻게 막을까?
- 공격자는 현대 컴퓨터의 빠른 연산 속도를 무기로 공격을 시도하고 있음
- 결국 해결책은 '연산 속도를 늦추는 것'이 핵심
- 연산 속도를 늦추기 위해서 의도적으로 비밀번로 검증 과정을 느리게 만듦
  - 느리게 맏르기 위해서 의도적으로 해시 연산을 수십만 번 반복시켜, 공격 속도를 늦춤

> 이 방식이 바로 "키 스트레칭(Key Stretching)"

```
연산 속도를 늦추면 사용자가 로그인할 때 느리다는 불편을 겪을 수도 있지 않을까요>
- 사용자는 실제로 체감하지 못합니다.
해시 연산 속도를 1개당 0.2초로 바꾼다고 하면, 실제로 공격자는 1초에 5번의 공격만 가능합니다.
하지만 사용자에게 0.2초는 느리다고 하기엔 충분히 빠른 시간입니다.
```


키 스트레칭(Key Stretching)
- 솔트를 적용한 해시 함수를 수만~수십만 번 반복하여 연산 시간을 의도적으로 늘리는 기법
- '속도'가 무기인 무차별 대입 공격(Brute-force Attack)을 방어할 수 있음
- Django는 이 키 스트레칭을 구현하기 위해서 PBKDF2 라는 검증된 알고리즘을 기본적으로 사용
  - 최근에는 더 강력한 보안을 제공하는 Argon2, bcrypt 같은 알고리즘도 지원

Django에서의 비밀번호 암호화

1. <algorithm>: 어떤 알고리즘을 쓰는 지
2. <iteratons>: 키 스트레칭 횟수
3. <salt>: 생성된 솔트
4. <hash>: 생성된 최종 해시

비밀번호 암호화 정리
- 암호화 과정을 이해한 후, 검증도니 프레임워크의 보안 기능을 신뢰하고 사용하기
  - 보안은 매우 어렵고 복잡한 분야이며, Django공식에도 가급적 재발명하지 않을 것을 권장하고 있음
  - 단순히 코드를 복사해서 붙여넣는 것을 넘어, 이 기능이 '왜'이렇게 만들어졌는지 이해하면, 더 견고하고 안전한 어플리케이션을 만들 수 있음
  - 사용자는 우리의 서비스를 믿고 소중한 개인정보를 맡기고, 그들의 데이터를 안전하게 지키는 것은 개발자의 '가장 기본적인 책임이자 윤리'

비밀번호 초기화

비밀번호 최기화
- 비밀번호를 잊어버린 사용자가 이메일을 활용하여 비밀번호를 다시 설정하는 과정
- 비밀번호 초기화 과정
  1. 비밀번호를 찾으려고 하는 이메일 잉ㅂ력
  2. 이메일로 비밀번호 재설정 링크를 전송
  3. 비밀번호 재설정 페이지에서 새로운 비밀번호 설정
  4. 초기화 후 다시 로그인

1. 비밀번호 찾으려고 하는 이메일 입력
   - 이메일을 입력하는 페이지를 직접 만들어야 할까?
     - NO
   - Django 에서는 비밀번호 관련된 다양한 기능을 관리자 페이지로 제공하고 있음
   - Django에서 제공하는 비밀번호 관련 기능을 활용하기 위해 django.contrib.auth.urls를 crud/urls.py에 포함
   - .../accounts/ 만 입력해보고, 제공하는 url 목록을 확인
   - 제공되는 url 목록에서 'password_reset' 경로를 입력해보고, 비밀번호 초기화 페이지로 접근

```
같은 accounts에 2개의 include가 사용되어도 괜찮나요?
- 내부 prefix로 여러 번 include를 해도, 각 include의 내부 URl 패턴을 순차적으로 모두 시도합니다.
- 내부 URl이 겹치지 않으면 모두 동작하기 때문에 괜찮습니다.
```

2. 이메일로 비밀번호 재설정 링크를 전송
   - 비밀번호를 찾으려고 하는 이메일을 작성하고 'Reset my password' 버튼을 눌러 이메일 전송

> 입력한 이멜일에 매칭되는 사용자 이메일이 없다면 이메일 기능은 동작하지 않음

```
왜 입력한 이메일에 매칭되는 사용자 이메일이 없다면 이메일 기능이 동작하지 않나요?
- 에러나 주의 문구가 나타나면 사용자 입장에서는 편리하겠지만, 임의의 사용자가 이메일의 존재 여부를 확인하는 용도로 악용하는 사례를 막기 위해서 동작하지 않습니다.
```

  - 이메일 관련 설정을 따로 하지 않았기 때문에 이메일 전송 실패
  - Django에서는 이메일을 보낸 내용을 콘솔에서 볼 수 있는 기능을 제공
  - settings.py에서 EMAIL_BACKEDN 세팅 후 다시 이메일 전송하기
  - 콘솔창에 비밀번호 초기화 설정 안내 이메일을 확인하고, 해당 링크 들어가기

3. 비밀번호 재설정 페이지에서 새로운 비밀번호 설정
   - 새로운 비밀번호 설정 후, 비밀번호 설정 완료 페이지 보여주기

4. 초기화 후 다시 로그인
   - 변경한 비밀번호로 로그인


Django 에서는 수많은 '완성된 기능 모듈'을 제공
- 단순하게 '비밀번호 초기화' 기능을 구현한 것이 아니라, 잘 만들어진 모듈을 활용하는 법을 학습한 것
  - Django.contrib.auth.urls 외에도 admin, sessions, stiemaps등 다양한 기능을 제공
  - 이메일 서버가 없어도 테스트가 가능한 것처럼, 실제 서비스 환경을 훙내 내는 많은 모듈을 제공하고 있음
- 이러한 다양한 모듈을 활용해서 개발 기술을 숙련하는 데 활용

PasswordChangeForm 인자 순서
- PasswordChangeForm이 다른 Form과 달리 user 객체를 첫번째 인자로 받는 이유
  - 부모 클래스인 SetPasswordForm의 생성자 함수 구성을 따르기 때문